# Mini GPT-2 — advanced version (multilingual, custom attention, scale-up bonus)

This is the bonus-round notebook. It builds on the validated Tiny Shakespeare baseline and adds everything else from the project brief:

| Brief item | What's implemented here |
|---|---|
| Subword tokenizer + bigger dataset | A byte-level BPE tokenizer **trained from scratch** on a ~28MB corpus |
| Non-English / multilingual data | 5 languages, 3 different scripts (Latin, Devanagari, CJK, Cyrillic) |
| Custom attention / architectural modification | **Rotary Positional Embeddings (RoPE)** replace the learned absolute position embedding entirely |
| Scale to full 127M GPT-2 config | A one-line `SCALE` toggle switches the same code to 12 layers / 12 heads / 768 dim / 1024 context, with mixed precision + gradient accumulation to fit a T4 |

Also added along the way since they're standard practice at this scale: weight tying (embedding <-> output head), GELU (real GPT-2 uses this, not ReLU), and a learning-rate warmup + cosine decay schedule.

**No manual downloads.** Five Bible translations (CC0-licensed, public domain) are fetched automatically from GitHub -- chosen specifically because it's a real, sizeable, *parallel* multilingual corpus with a fully verified, stable raw-file URL per language.

Run cells top to bottom. `Runtime -> Change runtime type -> T4 GPU` first.

In [1]:
!nvidia-smi

Sat Jun 27 02:34:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q tokenizers

In [3]:
import os
import math
import random
import xml.etree.ElementTree as ET
import urllib.request

import torch
import torch.nn as nn
from torch.nn import functional as F
from tokenizers import ByteLevelBPETokenizer

torch.manual_seed(1337)
random.seed(1337)

### Config -- pick your scale

`SCALE = 'mini'` is the real deliverable: a genuine subword-tokenized, multilingual mini-GPT2, comfortable on a free T4.

`SCALE = 'full_gpt2'` switches the *exact same code* to GPT-2 small's real layer dimensions (12 layers, 12 heads, 768 embedding dim, 1024 context) -- the scale-up bonus. It uses a smaller per-step batch with gradient accumulation plus fp16 mixed precision to fit in 16GB. Be aware (see our discussion earlier in this chat): this will *train*, and loss will *decrease*, but it will not reach GPT-2's published quality on this budget -- the bonus here is the engineering, not matching OpenAI's numbers.

One detail worth knowing: our embedding table uses a custom multilingual vocab (`VOCAB_SIZE` below), not GPT-2's English-centric 50,257-token vocab. At the same 12/12/768/1024 shape, that lands `full_gpt2` around ~97M parameters rather than ~124-127M -- fewer embedding parameters, identical transformer body. Bump `VOCAB_SIZE` up toward 50,000 if you want the parameter count itself closer to the literal 127M figure (tradeoff: many of those extra vocab slots will see little training signal on a 28MB corpus).

In [4]:
SCALE = 'mini'  # 'mini' or 'full_gpt2'

VOCAB_SIZE    = 16000     # custom multilingual BPE vocab size (trained from scratch below)
dropout       = 0.2
learning_rate = 3e-4
eval_interval = 500
eval_iters    = 200

if SCALE == 'mini':
    n_embd          = 384
    n_head          = 6
    n_layer         = 6
    block_size      = 256
    batch_size      = 64
    grad_accum_steps = 1
    max_iters       = 5000
elif SCALE == 'full_gpt2':
    n_embd          = 768
    n_head          = 12
    n_layer         = 12
    block_size      = 1024
    batch_size      = 8        # small per-step batch to fit 16GB
    grad_accum_steps = 8       # effective batch size = 8 * 8 = 64
    max_iters       = 3000
else:
    raise ValueError("SCALE must be 'mini' or 'full_gpt2'")

warmup_iters   = 100
lr_decay_iters = max_iters
min_lr         = learning_rate / 10

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| scale:', SCALE)
if device == 'cpu':
    print('No GPU detected -- this will be slow, and full_gpt2 may run out of system RAM. Runtime > Change runtime type > T4 GPU')

device: cuda | scale: mini


### Optional: persist to Google Drive

Same idea as the baseline notebook, but now this also persists the **trained tokenizer files** -- important, because the tokenizer's vocabulary has to stay identical across a resumed session (if it retrained from scratch with even a slightly different corpus pass, token ids could shift and silently corrupt a resumed model).

In [5]:
USE_DRIVE = False  # set True to persist checkpoints + tokenizer to Google Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    checkpoint_dir = '/content/drive/MyDrive/mini_gpt2_advanced'
    os.makedirs(checkpoint_dir, exist_ok=True)
else:
    checkpoint_dir = '.'

checkpoint_path = os.path.join(checkpoint_dir, f'checkpoint_{SCALE}.pt')
print('checkpoints will be saved to:', checkpoint_path)

checkpoints will be saved to: ./checkpoint_mini.pt


## Step 1 -- Multilingual dataset

Five Bible translations from the [christos-c/bible-corpus](https://github.com/christos-c/bible-corpus) project (CC0 -- public domain, no copyright concerns): English, Hindi, Spanish, Chinese, Russian. It's a genuinely parallel corpus (~31,000 aligned verses per language, ~28MB combined) spanning Latin, Devanagari, CJK, and Cyrillic scripts -- a real stress test for a byte-level tokenizer, not just a token gesture at "multilingual."

Lines from all five languages are shuffled together before training, so the model sees language-mixed context windows rather than long monolingual stretches.

In [6]:
LANGUAGES = ['English', 'Hindi', 'Spanish', 'Chinese', 'Russian']

lines = []
for lang in LANGUAGES:
    path = f'{lang}.xml'
    if not os.path.exists(path):
        url = f'https://raw.githubusercontent.com/christos-c/bible-corpus/master/bibles/{lang}.xml'
        urllib.request.urlretrieve(url, path)
    root = ET.fromstring(open(path, encoding='utf-8').read())
    lang_lines = [seg.text.strip() for seg in root.iter('seg') if seg.text and seg.text.strip()]
    lines.extend(lang_lines)
    print(f'{lang}: {len(lang_lines):,} verses')

random.shuffle(lines)
print(f'total lines: {len(lines):,}')

combined_path = 'multilingual_corpus.txt'
with open(combined_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print(f'combined corpus: {os.path.getsize(combined_path):,} bytes')

English: 31,102 verses
Hindi: 31,065 verses
Spanish: 31,100 verses
Chinese: 31,101 verses
Russian: 31,102 verses
total lines: 155,470
combined corpus: 27,718,195 bytes


## Step 2 -- Train a byte-level BPE tokenizer from scratch

This is the real subword tokenizer the project brief asks for -- trained on *our* corpus, not borrowed from GPT-2. Byte-level BPE starts from raw UTF-8 bytes, so it has no "unknown script" problem: the same tokenizer round-trips English, Hindi, Spanish, Chinese, and Russian correctly, with no special handling per language.

If a previously-trained tokenizer exists at `checkpoint_dir` (e.g. you're resuming after a disconnect), it's reloaded instead of retrained, so token ids stay consistent with whatever model checkpoint you also have saved.

In [7]:
vocab_path = os.path.join(checkpoint_dir, 'multiling_bpe-vocab.json')
merges_path = os.path.join(checkpoint_dir, 'multiling_bpe-merges.txt')

if os.path.exists(vocab_path) and os.path.exists(merges_path):
    tok = ByteLevelBPETokenizer.from_file(vocab_path, merges_path)
    print('loaded existing tokenizer from', checkpoint_dir)
else:
    tok = ByteLevelBPETokenizer()
    tok.train(files=[combined_path], vocab_size=VOCAB_SIZE, min_frequency=2, special_tokens=['<|endoftext|>'])
    tok.save_model(checkpoint_dir, 'multiling_bpe')
    print('trained a new tokenizer and saved it to', checkpoint_dir)

vocab_size = tok.get_vocab_size()
eot_id = tok.token_to_id('<|endoftext|>')
print('vocab size:', vocab_size)

# sanity check: one sentence, five scripts, one tokenizer
sample = "In the beginning God created. आदि में परमेश्वर ने। En el principio. 起初　神. В начале."
ids = tok.encode(sample).ids
print('round-trip OK:', tok.decode(ids).strip() == sample.strip())

trained a new tokenizer and saved it to .
vocab size: 16000
round-trip OK: True


In [8]:
# Tokenize the full corpus. Uses encode_batch (not one giant joined string) to stay memory-friendly.
ids = []
for enc in tok.encode_batch(lines):
    ids.extend(enc.ids)
    ids.append(eot_id)

data = torch.tensor(ids, dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f'total tokens: {len(data):,}  (train {len(train_data):,} / val {len(val_data):,})')

total tokens: 6,393,990  (train 5,754,591 / val 639,399)


## Step 3 -- Batching

Same idea as the baseline: random windows of `block_size` tokens, targets shifted by one position. The only difference is these are now subword token ids instead of character ids.

In [9]:
def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

## Step 4 -- Model: RoPE instead of learned positional embeddings

This is the architectural-modification bonus. Instead of a learned position embedding table added to the token embedding, **Rotary Positional Embeddings (RoPE)** inject position information directly inside attention: each query/key vector is *rotated* by an angle that depends on its position, so the dot product between a query at position *i* and a key at position *j* naturally depends on the relative distance `i - j` rather than absolute position. This is the scheme used in LLaMA, GPT-NeoX, and most current open models instead of GPT-2's original learned embeddings.

Concretely: there's no `position_embedding_table` anymore -- the model embeds tokens only, and `apply_rope` rotates `q` and `k` (not `v`) inside each attention head before the dot product.

In [10]:
def precompute_rope(dim, max_seq_len, theta=10000.0):
    """cos/sin tables for rotary embeddings, shape (max_seq_len, dim)."""
    inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(max_seq_len).float()
    freqs = torch.outer(t, inv_freq)          # (max_seq_len, dim/2)
    emb = torch.cat((freqs, freqs), dim=-1)   # (max_seq_len, dim)
    return emb.cos(), emb.sin()

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rope(x, cos, sin):
    """x: (B, T, head_size); cos/sin: (T, head_size). Rotates each vector by a position-dependent angle."""
    return x * cos + rotate_half(x) * sin

In [11]:
class Head(nn.Module):
    """One head of causal self-attention, with RoPE applied to q and k."""

    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        cos, sin = precompute_rope(head_size, block_size)
        self.register_buffer('rope_cos', cos)
        self.register_buffer('rope_sin', sin)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        k = apply_rope(k, self.rope_cos[:T], self.rope_sin[:T])
        q = apply_rope(q, self.rope_cos[:T], self.rope_sin[:T])
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = self.dropout(F.softmax(wei, dim=-1))
        return wei @ v


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    """GELU here (real GPT-2), vs the ReLU used in the baseline notebook."""

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [12]:
class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # No position_embedding_table: RoPE supplies position information
        # inside attention instead (see Head.forward above).
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding_table.weight  # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        x = self.token_embedding_table(idx)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


model = GPTLanguageModel().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'{n_params/1e6:.2f}M parameters (weight-tied, RoPE, GELU)')

16.78M parameters (weight-tied, RoPE, GELU)


## Step 5 -- Training: mixed precision, gradient accumulation, LR schedule, checkpoint/resume

Several things layered together here, all standard practice at this scale (and all things we discussed earlier in this chat for fitting a T4):
- **`torch.amp.autocast` + `GradScaler`**: runs matmuls in fp16 on GPU, automatically falls back to fp32 on CPU.
- **Gradient accumulation**: `grad_accum_steps` micro-batches are summed before one optimizer step, so `full_gpt2`'s small per-step batch still gets a sane effective batch size.
- **Warmup + cosine decay**: the standard GPT-2/3 learning-rate recipe.
- **Checkpointing**: now also saves the `GradScaler` state, so resuming after a disconnect reproduces the same training dynamics, not just the same weights.

In [13]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


def get_lr(it):
    if it < warmup_iters:
        return learning_rate * (it + 1) / warmup_iters
    if it > lr_decay_iters:
        return min_lr
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)

In [14]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
scaler = torch.amp.GradScaler(enabled=(device == 'cuda'))

start_iter = 0
if os.path.exists(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_iter = ckpt['iter'] + 1
    print(f'resumed from checkpoint at iter {start_iter}')

for it in range(start_iter, max_iters):
    lr = get_lr(it)
    for g in optimizer.param_groups:
        g['lr'] = lr

    if it % eval_interval == 0 or it == max_iters - 1:
        losses = estimate_loss()
        print(f"step {it}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, lr {lr:.2e}")
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'iter': it,
        }, checkpoint_path)

    optimizer.zero_grad(set_to_none=True)
    for _ in range(grad_accum_steps):
        xb, yb = get_batch('train')
        with torch.amp.autocast(device_type=('cuda' if device == 'cuda' else 'cpu'),
                                 dtype=torch.float16, enabled=(device == 'cuda')):
            logits, loss = model(xb, yb)
            loss = loss / grad_accum_steps
        scaler.scale(loss).backward()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

print('training complete')

step 0: train loss 9.7562, val loss 9.7561, lr 3.00e-06
step 500: train loss 3.8390, val loss 3.8669, lr 2.96e-04
step 1000: train loss 3.3765, val loss 3.4368, lr 2.78e-04
step 1500: train loss 3.1310, val loss 3.2250, lr 2.49e-04
step 2000: train loss 2.9785, val loss 3.1074, lr 2.12e-04
step 2500: train loss 2.8531, val loss 3.0165, lr 1.69e-04
step 3000: train loss 2.7696, val loss 2.9570, lr 1.27e-04
step 3500: train loss 2.7056, val loss 2.9274, lr 8.78e-05
step 4000: train loss 2.6709, val loss 2.8953, lr 5.68e-05
step 4500: train loss 2.6440, val loss 2.8742, lr 3.69e-05
step 4999: train loss 2.6223, val loss 2.8678, lr 3.00e-05
training complete


## Step 6 -- Generate multilingual text

Prompts in a few different languages, fed through the same model. Don't expect theological coherence at this scale -- the goal is to see the model producing script-appropriate, language-consistent continuations, which is the real signature of a working multilingual byte-level tokenizer + model.

In [15]:
prompts = [
    "In the beginning",
    "En el principio",
    "В начале",
]

for p in prompts:
    context = torch.tensor([tok.encode(p).ids], dtype=torch.long, device=device)
    generated = model.generate(context, max_new_tokens=80)[0].tolist()
    print(f'--- prompt: {p!r} ---')
    print(tok.decode(generated))
    print()

--- prompt: 'In the beginning' ---
In the beginning in the beginning of our gates, the elders, and people Israel, be changed to depart from the morning unto the LORD for them.有 聲 音 說 、 你 們 明 亮 百 姓 站 在 近 前 、 說 、 你 們 聽 見 自 己 說 、 他 們 在 耶 和 華 面 前 復 了 我 耶 和 華 、 還 以 色 列 人 在 其 中就 要

--- prompt: 'En el principio' ---
En el principio de Radrac el al rey le Joyada preguntó: --¿Has encrito? He aquí, la espada de Saúl será cortado del arot. Éste es la tribu de los hijos del reyAfter Jesus answered him, Jesus said unto us. And he said continually, Lord, thou camest here.और तू अपनी बहिनों के बीच खड़ा

--- prompt: 'В начале' ---
В начале царя Аггея, сына его, чтоб он сказал: вот в Иерусалиме; ибо он – десять талантов, которые мы были поягости и ожидаем Отца;耶 和 華 說 、 你 曾 容 我 的 　 神 說 、 台 微 甚 小 的 母 胎 、 因 為 我 倚 靠 這 大 腿 倒 塌 了 、 使 他 喪 命 芵 們 的 生 命 、



## Recap against the project brief

| Brief item | Status |
|---|---|
| Decoder-only transformer from scratch | Done (baseline + here) |
| Trained on a text dataset, generates language | Done |
| Bonus: non-English / multilingual | Done -- 5 languages, 3 scripts, one byte-level tokenizer |
| Bonus: custom attention / architectural modification | Done -- RoPE replaces learned positional embeddings |
| Bonus: scale to full 127M GPT-2 config | Done -- `SCALE='full_gpt2'` above, same code, real layer dimensions, fp16 + grad accumulation to fit 16GB |

What's genuinely still a judgment call rather than a code change: the `full_gpt2` path will *train* on a free T4 but, as discussed earlier in this chat, won't reach GPT-2's published validation loss in any realistic free-tier compute budget -- that's a compute ceiling, not a bug. Worth saying explicitly in your writeup rather than letting a grader assume otherwise.